# Main Benchmark Notebook\n
\n
This notebook will contain the end-to-end benchmark pipeline in later steps.\n

In [ ]:
!git clone https://github.com/JunyangHe/timeseries_benchmark_demo.git
%cd timeseries_benchmark_demo
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import zipfile

import pandas as pd

CSV_FILENAME = "BasicInputTimeSeries_us.csv"

# Locate zip in either local repo execution or Colab-cloned execution.
ZIP_PATH_CANDIDATES = [
    Path("data/raw/BasicInputTimeSeries_us.zip"),
    Path("/content/timeseries_benchmark_demo/data/raw/BasicInputTimeSeries_us.zip"),
]

zip_path = next((p for p in ZIP_PATH_CANDIDATES if p.exists()), None)
if zip_path is None:
    raise FileNotFoundError(
        "Could not find BasicInputTimeSeries_us.zip in expected paths. "
        "Update ZIP_PATH_CANDIDATES to match your environment."
    )

extract_dir = zip_path.parent
csv_path = extract_dir / CSV_FILENAME

# Unzip before loading. If CSV already exists, skip extraction.
if not csv_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        if CSV_FILENAME not in zf.namelist():
            raise FileNotFoundError(
                f"{CSV_FILENAME} not found inside zip archive: {zip_path}"
            )
        zf.extract(CSV_FILENAME, path=extract_dir)

# Load only the required columns and parse the date column.
# basin_id is kept as string to preserve identifier semantics.
df = pd.read_csv(
    csv_path,
    usecols=["Year_Mnth_Day", "basin_id", "PRCP(mm/day)", "Tmean(C)", "QObs(mm/d)"],
    parse_dates=["Year_Mnth_Day"],
    dtype={
        "basin_id": "string",
        "PRCP(mm/day)": "float32",
        "Tmean(C)": "float32",
        "QObs(mm/d)": "float32",
    },
)

# Essential formatting: standardized column names, ordering, and basic checks.
df = df.rename(
    columns={
        "Year_Mnth_Day": "date",
        "PRCP(mm/day)": "prcp_mm_day",
        "Tmean(C)": "tmean_c",
        "QObs(mm/d)": "qobs_mm_d",
    }
)

df = df.sort_values(["basin_id", "date"]).reset_index(drop=True)

TARGET_COLUMN = "qobs_mm_d"

required_cols = ["date", "basin_id", "prcp_mm_day", "tmean_c", TARGET_COLUMN]
missing_values = df[required_cols].isna().sum()
if missing_values.any():
    raise ValueError(f"Found missing values in required columns:\n{missing_values}")

print(f"Zip source: {zip_path}")
print(f"CSV loaded from: {csv_path}")
print(f"Loaded rows: {len(df):,}")
print(f"Unique basins: {df['basin_id'].nunique():,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Target column: {TARGET_COLUMN}")

df.head()

In [ ]:
import matplotlib.pyplot as plt

# =========================
# Sanity checks (preprocessing)
# =========================

# 1) Shape / dtypes / head / tail
print("=== Shape ===")
print(df.shape)

print("\n=== Dtypes ===")
print(df.dtypes)

print("\n=== Head ===")
display(df.head())

print("\n=== Tail ===")
display(df.tail())

if not pd.api.types.is_datetime64_any_dtype(df["date"]):
    raise TypeError("Column 'date' is not datetime. Ensure parse_dates worked correctly.")

# 2) Summary stats (check impossible min/max or odd spread)
print("\n=== Summary Statistics (numeric columns) ===")
summary = df[["prcp_mm_day", "tmean_c", "qobs_mm_d"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
display(summary)

# Optional guardrails for clearly impossible values in this dataset context
if (df["prcp_mm_day"] < 0).any():
    print("Warning: Negative precipitation values found.")
if (df["qobs_mm_d"] < 0).any():
    print("Warning: Negative observed streamflow values found.")

# 3) Missing values
print("\n=== Missing Values by Column ===")
missing_counts = df.isna().sum().sort_values(ascending=False)
display(missing_counts)

# 4) Date span and calendar continuity
print("\n=== Date Span ===")
min_date = df["date"].min()
max_date = df["date"].max()
print(f"Min date: {min_date}")
print(f"Max date: {max_date}")

# Use unique dates across all basins to check global day coverage
unique_dates = pd.Index(sorted(df["date"].dt.normalize().unique()))
expected_dates = pd.date_range(start=min_date.normalize(), end=max_date.normalize(), freq="D")
missing_dates = expected_dates.difference(unique_dates)
print(f"Total unique dates in data: {len(unique_dates):,}")
print(f"Expected daily dates in span: {len(expected_dates):,}")
print(f"Missing calendar dates (global): {len(missing_dates):,}")

# 5) Time plot of target over full series (aggregated across basins)
# Mean and median by day help reveal gaps, spikes, and flat segments.
daily_target = (
    df.groupby("date", as_index=False)["qobs_mm_d"]
    .agg(["mean", "median", "count"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_target["date"], daily_target["mean"], label="Daily mean qobs_mm_d", linewidth=1.0)
ax.plot(daily_target["date"], daily_target["median"], label="Daily median qobs_mm_d", linewidth=1.0, alpha=0.8)
ax.set_title("Target Over Time (Aggregated Across Basins)")
ax.set_xlabel("Date")
ax.set_ylabel("QObs (mm/d)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Optional supporting plot: number of basin records per day (coverage consistency)
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(daily_target["date"], daily_target["count"], color="tab:orange", linewidth=1.0)
ax.set_title("Daily Record Count (Coverage Check)")
ax.set_xlabel("Date")
ax.set_ylabel("Row count")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 6) Histogram of target distribution
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df["qobs_mm_d"], bins=100)
ax.set_title("Target Distribution: qobs_mm_d")
ax.set_xlabel("QObs (mm/d)")
ax.set_ylabel("Frequency")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

# =========================
# Preprocessing
# =========================

TARGET_STREAM = "qobs_mm_d"
FEATURE_STREAMS = ["prcp_mm_day", "tmean_c"]
ALL_STREAMS = FEATURE_STREAMS + [TARGET_STREAM]

# Ensure stable ordering before group-wise operations.
df_preprocessed = df.sort_values(["basin_id", "date"]).copy()

# 1) Forward imputation within each basin (time-aware).
# This fills missing values using previous observations from the same basin.
df_preprocessed[ALL_STREAMS] = (
    df_preprocessed.groupby("basin_id", group_keys=False)[ALL_STREAMS].ffill()
)

# 2) Remove rows still empty after forward fill (e.g., leading missing blocks).
before_drop = len(df_preprocessed)
df_preprocessed = df_preprocessed.dropna(subset=ALL_STREAMS).reset_index(drop=True)
after_drop = len(df_preprocessed)

# 3) Scale with StandardScaler.
# Keep qobs_mm_d as the target stream; create scaled columns for modeling use.
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

df_preprocessed[[f"{col}_scaled" for col in FEATURE_STREAMS]] = feature_scaler.fit_transform(
    df_preprocessed[FEATURE_STREAMS]
)
df_preprocessed[f"{TARGET_STREAM}_scaled"] = target_scaler.fit_transform(
    df_preprocessed[[TARGET_STREAM]]
)

print("=== Preprocessing Complete ===")
print(f"Target stream: {TARGET_STREAM}")
print(f"Feature streams: {FEATURE_STREAMS}")
print(f"Rows before dropna: {before_drop:,}")
print(f"Rows after dropna:  {after_drop:,}")
print(f"Rows removed:       {before_drop - after_drop:,}")

display(
    df_preprocessed[
        [
            "date",
            "basin_id",
            *FEATURE_STREAMS,
            TARGET_STREAM,
            *(f"{c}_scaled" for c in FEATURE_STREAMS),
            f"{TARGET_STREAM}_scaled",
        ]
    ].head()
)

In [ ]:
# Zero-shot forecasting with Chronos-2 (univariate qobs)
# Uses previously created df_preprocessed

import numpy as np
import pandas as pd

# Install once per runtime when needed (Colab/local notebook runtime)
# %pip install -q "chronos-forecasting>=2.0"

from chronos import Chronos2Pipeline
import torch

LOOKBACK_WINDOW = 30
FORECAST_HORIZON = 1
TARGET_COL = "qobs_mm_d"
ID_COL = "basin_id"
TIME_COL = "date"

if "df_preprocessed" not in globals():
    raise NameError("df_preprocessed is not defined. Run the loading and preprocessing cells first.")

# Univariate modeling frame (target stream only)
model_df = (
    df_preprocessed[[TIME_COL, ID_COL, TARGET_COL]]
    .dropna(subset=[TIME_COL, ID_COL, TARGET_COL])
    .sort_values([ID_COL, TIME_COL])
    .reset_index(drop=True)
)

# Build one zero-shot forecast per basin for the final available day.
# Context = previous 30 days, target = last day (1-step ahead from context end).
context_frames = []
eval_rows = []

for basin_id, g in model_df.groupby(ID_COL, sort=False):
    g = g.sort_values(TIME_COL).reset_index(drop=True)

    if len(g) < LOOKBACK_WINDOW + FORECAST_HORIZON:
        continue

    context_slice = g.iloc[-(LOOKBACK_WINDOW + FORECAST_HORIZON):-FORECAST_HORIZON][
        [ID_COL, TIME_COL, TARGET_COL]
    ].copy()

    target_row = g.iloc[-FORECAST_HORIZON].copy()

    context_frames.append(context_slice)
    eval_rows.append(
        {
            ID_COL: str(target_row[ID_COL]),
            TIME_COL: target_row[TIME_COL],
            "y_true": float(target_row[TARGET_COL]),
        }
    )

if not context_frames:
    raise ValueError("No eligible basins for forecasting. Check data length after preprocessing.")

context_df = pd.concat(context_frames, ignore_index=True)
context_df[ID_COL] = context_df[ID_COL].astype(str)

expected_context_rows = len(eval_rows) * LOOKBACK_WINDOW
if len(context_df) != expected_context_rows:
    raise ValueError(
        f"Context row mismatch: expected {expected_context_rows}, got {len(context_df)}"
    )

# Chronos-2 pipeline
# Device auto-selection: CUDA if available, otherwise CPU
device_map = "cuda" if torch.cuda.is_available() else "cpu"
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map=device_map)

pred_df = pipeline.predict_df(
    context_df=context_df,
    prediction_length=FORECAST_HORIZON,
    quantile_levels=[0.5],
    id_column=ID_COL,
    timestamp_column=TIME_COL,
    target=TARGET_COL,
)

# Resolve the median prediction column robustly.
pred_col = None
if "0.5" in pred_df.columns:
    pred_col = "0.5"
else:
    candidate_cols = [
        c for c in pred_df.columns if str(c).strip().lower() in {"0.5", "median", "mean"}
    ]
    if candidate_cols:
        pred_col = candidate_cols[0]

if pred_col is None:
    raise KeyError(
        f"Could not find forecast column in predictions. Available columns: {list(pred_df.columns)}"
    )

pred_out = pred_df[[ID_COL, TIME_COL, pred_col]].copy()
pred_out[ID_COL] = pred_out[ID_COL].astype(str)
pred_out = pred_out.rename(columns={pred_col: "y_pred"})

eval_df = pd.DataFrame(eval_rows)
eval_df[ID_COL] = eval_df[ID_COL].astype(str)

results_df = eval_df.merge(pred_out, on=[ID_COL, TIME_COL], how="inner")

if results_df.empty:
    raise ValueError(
        "No matched predictions for evaluation. "
        "Check prediction timestamps and columns returned by Chronos-2."
    )

rmse = float(np.sqrt(np.mean((results_df["y_true"] - results_df["y_pred"]) ** 2)))

print("=== Chronos-2 Zero-shot Evaluation ===")
print(f"Device: {device_map}")
print(f"Target stream: {TARGET_COL}")
print(f"Input mode: univariate")
print(f"Lookback window: {LOOKBACK_WINDOW}")
print(f"Forecast horizon: {FORECAST_HORIZON}")
print(f"Evaluated basins: {results_df[ID_COL].nunique():,}")
print(f"Evaluated points: {len(results_df):,}")
print(f"RMSE: {rmse:.6f}")

display(results_df.head())